In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [4]:
docs[0].metadata

{'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [6]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [7]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

generator_llm = ChatOpenAI(model="gpt-4o-mini")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [8]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [
    SummaryExtractor(llm=generator.llm),

    EmbeddingExtractor(
        embedding_model=generator.embedding_model,
        property_name ="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm),
]

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),
]

In [10]:
#테스트셋 생성
testset = generator.generate_with_langchain_docs(
    documents =docs,
    testset_size=10,
    transforms=transforms,
    query_distribution=query_distribution,
    raise_exceptions=True,
)

Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]